In [53]:
%pip install meteostat



[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [54]:
from datetime import date
from meteostat import Point, daily
import pandas as pd
import numpy as np

In [55]:
##Loading data from meteostat from Orly weather station based on station ID
STATION_ID = '07149'
orly = ms.Station(id=STATION_ID)

# Use a short range that is definitely within the inventory
start = date(1995, 1, 1)
end   = date(2025, 1, 31)

ts = ms.daily(orly, start, end)
df = ts.fetch()

print(df.head())


            temp  tmin  tmax  rhum  prcp  snwd  wspd  wpgt  pres  tsun  cldc
time                                                                        
1995-01-01   2.9   1.7   4.5  <NA>   0.4  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>
1995-01-02   1.4  -0.6   3.3  <NA>   0.0  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>
1995-01-03   1.5  -0.3   5.1  <NA>   0.0  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>
1995-01-04  -1.1  -2.8   2.3  <NA>   0.0  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>
1995-01-05  -3.1  -5.9  -1.4  <NA>   5.2  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>


### Check, clean and prepare df

In [56]:
print(df.dtypes)
print(df.isna().sum())
## we can see that there are missing values in avg temperature

temp    Float64
tmin    Float64
tmax    Float64
rhum      UInt8
prcp    Float64
snwd     UInt16
wspd    Float64
wpgt    Float64
pres    Float64
tsun     UInt16
cldc      UInt8
dtype: object
temp      366
tmin        0
tmax        0
rhum     1967
prcp        0
snwd    10862
wspd     1961
wpgt     8614
pres     2250
tsun    10094
cldc     4223
dtype: int64


In [57]:
# Keep only the columns we need
df = df[['tmin', 'tmax']].copy()

# Ensure numeric types
df['tmin'] = pd.to_numeric(df['tmin'], errors='coerce')
df['tmax'] = pd.to_numeric(df['tmax'], errors='coerce')

# Sort & ensure datetime index
df = df.sort_index()
print(df.head())



            tmin  tmax
time                  
1995-01-01   1.7   4.5
1995-01-02  -0.6   3.3
1995-01-03  -0.3   5.1
1995-01-04  -2.8   2.3
1995-01-05  -5.9  -1.4


In [58]:
## calculate average ourselves, to ensure no nulls based on the CME style 
df['tavg'] = (df['tmin'] + df['tmax']) / 2
print(df.head())

            tmin  tmax  tavg
time                        
1995-01-01   1.7   4.5   3.1
1995-01-02  -0.6   3.3  1.35
1995-01-03  -0.3   5.1   2.4
1995-01-04  -2.8   2.3 -0.25
1995-01-05  -5.9  -1.4 -3.65


In [59]:
print(df[['tmin', 'tmax', 'tavg']].describe())
print(df.isna().sum())


           tmin       tmax       tavg
count   10989.0    10989.0    10989.0
mean   8.117108  16.514114  12.315611
std    5.882536   7.818493    6.62431
min       -13.3       -7.6       -9.7
25%         3.9       10.6       7.45
50%         8.3       16.4      12.25
75%        12.7       22.5      17.45
max        24.4       41.9      33.15
tmin    0
tmax    0
tavg    0
dtype: int64


### compute values needed for HBA

In [60]:
EURO_PER_POINT = 20  # €20 per index point (CME spec)

# 1. Compute HDD & CDD if not already present
if 'tavg' not in df.columns:
    df['tavg'] = (df['tmin'] + df['tmax']) / 2

df['hdd'] = (18 - df['tavg']).clip(lower=0)
df['cdd'] = (df['tavg'] - 18).clip(lower=0)
df['cat'] = df['tavg'].clip(lower=0)

# 2. Monthly indices
monthly_hdd = df['hdd'].resample('ME').sum()
monthly_cdd = df['cdd'].resample('ME').sum()
monthly_cat = df['cat'].resample('ME').sum()

# 3. Convert to € settlement per contract
monthly_hdd_eur = monthly_hdd * EURO_PER_POINT
monthly_cdd_eur = monthly_cdd * EURO_PER_POINT
monthly_cat_eur = monthly_cat * EURO_PER_POINT



In [65]:
#create a loop for all 12 months of the year to calculate burn stats
rows_hdd = []
rows_cdd = []
rows_cat = []

for month in range(1, 12 + 1):
    # --- HDD stats ---
    hdd_vals = monthly_hdd_eur[monthly_hdd_eur.index.month == month]
    
    if len(hdd_vals) > 0:
        rows_hdd.append({
            "month": month,
            "n_years": len(hdd_vals),
            "mean_burn": hdd_vals.mean(),          # expected payout (burn price)
            "std": hdd_vals.std(),                 # volatility
            "min": hdd_vals.min(),
            "p10": hdd_vals.quantile(0.10),
            "median": hdd_vals.median(),
            "p90": hdd_vals.quantile(0.90),
            "max": hdd_vals.max(),
        })
    else:
        rows_hdd.append({
            "month": month,
            "n_years": 0,
            "mean_burn": np.nan,
            "std": np.nan,
            "min": np.nan,
            "p10": np.nan,
            "median": np.nan,
            "p90": np.nan,
            "max": np.nan,
        })

    # --- CDD stats ---
    cdd_vals = monthly_cdd_eur[monthly_cdd_eur.index.month == month]
    
    if len(cdd_vals) > 0:
        rows_cdd.append({
            "month": month,
            "n_years": len(cdd_vals),
            "mean_burn": cdd_vals.mean(),
            "std": cdd_vals.std(),
            "min": cdd_vals.min(),
            "p10": cdd_vals.quantile(0.10),
            "median": cdd_vals.median(),
            "p90": cdd_vals.quantile(0.90),
            "max": cdd_vals.max(),
        })
    else:
        rows_cdd.append({
            "month": month,
            "n_years": 0,
            "mean_burn": np.nan,
            "std": np.nan,
            "min": np.nan,
            "p10": np.nan,
            "median": np.nan,
            "p90": np.nan,
            "max": np.nan,
        })
# --- CAT stats ---
    cat_vals = monthly_cat_eur[monthly_cat_eur.index.month == month]

    rows_cat.append({
        "month": month,
        "n_years": len(cat_vals),
        "mean_burn": cat_vals.mean(),
        "std": cat_vals.std(),
        "min": cat_vals.min(),
        "p10": cat_vals.quantile(0.10),
        "median": cat_vals.median(),
        "p90": cat_vals.quantile(0.90),
        "max": cat_vals.max(),
    })

# Turn into DataFrames
burn_hdd = pd.DataFrame(rows_hdd).set_index("month")
burn_cdd = pd.DataFrame(rows_cdd).set_index("month")
burn_cat = pd.DataFrame(rows_cat).set_index("month")

print("HDD burn stats (€/contract):")
print(burn_hdd)

print("\nCDD burn stats (€/contract):")
print(burn_cdd)

print("\nCAT burn stats (€/contract):")
print(burn_cat)



HDD burn stats (€/contract):
       n_years    mean_burn          std     min     p10  median     p90  \
month                                                                      
1           31  8280.129032  1100.000174  6445.0  7124.0  8234.0  9976.0   
2           30  6920.066667  1205.646999  5176.0  5469.5  6716.0  8573.3   
3           30  5954.666667   834.504327  4688.0  4859.0  5877.5  6939.8   
4           30  4018.466667   873.430720  1992.0  2797.2  4082.5  4973.1   
5           30  2127.500000   695.892617   967.0  1327.8  2010.0  3092.3   
6           30   653.400000   328.569271     7.0   242.4   661.5  1029.1   
7           30   215.733333   208.420155     0.0     8.1   166.0   482.2   
8           30   215.600000   172.188469     0.0    53.0   173.5   494.5   
9           30  1154.466667   585.709221   237.0   438.9  1074.0  1986.1   
10          30  3093.466667   933.817369  1387.0  1827.1  3263.5  4089.9   
11          30  5947.366667   831.468351  4321.0  4878.2  6

### HDD vs 2027 prices

In [63]:
# ----------------------
# HDD: listed months & 2027 prices
# ----------------------
#https://www.cmegroup.com/markets/weather/hdd/paris-hdd-monthly.settlements.html

# Listed HDD months on CME (Oct–Apr)
listed_hdd_months = [10, 11, 12, 1, 2, 3, 4]

# Subset burn table to listed months
hdd_listed = burn_hdd.loc[listed_hdd_months].copy()
hdd_listed['month_name'] = ['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr']
# HDD 2027 settlement prices (index points)
hdd_2027_prices = {
    10: 143.0,   # Oct 27
    11: 281.0,   # Nov 27
    12: 366.0,   # Dec 27
    1:  404.0,   # Jan 27
    2:  332.0,   # Feb 27
    3:  284.0,   # Mar 27
    4:  189.0,   # Apr 27
}

hdd_market_2027 = pd.DataFrame.from_dict(
    hdd_2027_prices, orient='index', columns=['settle_index_2027']
)
hdd_market_2027.index.name = 'month'

# Convert to € settlement
hdd_market_2027['market_eur_2027'] = (
    hdd_market_2027['settle_index_2027'] * EURO_PER_POINT
)

# Join with burn
hdd_comp_2027 = hdd_listed.join(hdd_market_2027)

# Market vs burn
hdd_comp_2027['market_minus_burn'] = (
    hdd_comp_2027['market_eur_2027'] - hdd_comp_2027['mean_burn']
)
hdd_comp_2027['market_over_burn'] = (
    hdd_comp_2027['market_eur_2027'] / hdd_comp_2027['mean_burn']
)

print("HDD 2027 vs burn (€/contract):")
print(hdd_comp_2027[
    ['month_name', 'n_years', 'mean_burn', 'market_eur_2027',
     'market_minus_burn', 'market_over_burn', 'p10', 'p90', 'max']
])



HDD 2027 vs burn (€/contract):
      month_name  n_years    mean_burn  market_eur_2027  market_minus_burn  \
month                                                                        
10           Oct       30  3093.466667           2860.0        -233.466667   
11           Nov       30  5947.366667           5620.0        -327.366667   
12           Dec       30  7880.300000           7320.0        -560.300000   
1            Jan       31  8280.129032           8080.0        -200.129032   
2            Feb       30  6920.066667           6640.0        -280.066667   
3            Mar       30  5954.666667           5680.0        -274.666667   
4            Apr       30  4018.466667           3780.0        -238.466667   

       market_over_burn     p10     p90      max  
month                                             
10             0.924529  1827.1  4089.9   5048.0  
11             0.944956  4878.2  6807.1   7816.0  
12             0.928899  6779.6  9061.1  10845.0  
1          

### CAT vs 2027 prices 

In [67]:
# ----------------------
# CAT: listed months & 2027 prices
# ----------------------
#https://www.cmegroup.com/markets/weather/cat/paris-cat-monthly.settlements.html

# Listed CAT months on CME (Europe summer)
listed_cdd_months = [4,5, 6, 7, 8, 9, 10]

cat_listed = burn_cdd.loc[listed_cdd_months].copy()
cat_listed['month_name'] = ['April','May', 'Jun', 'Jul', 'Aug', 'Sep','October']
# CAT 2027 settlement prices (index points)
cat_2027_prices = {
    4:  352.0,   # Apr 27
    5:  468.0,   # May 27
    6: 576.0,   # Jun 27
    7: 653.0,   # Jul 27
    8: 642.0,   # Aug 27
    9:  522.0,   # Sep 27
    10: 418.0,   # Oct 27
}

cat_market_2027 = pd.DataFrame.from_dict(
    cat_2027_prices, orient='index', columns=['settle_index_2027']
)
cat_market_2027.index.name = 'month'

# Convert to € settlement
cat_market_2027['market_eur_2027'] = (
    cat_market_2027['settle_index_2027'] * EURO_PER_POINT
)

# Join with burn
cat_comp_2027 = cat_listed.join(cat_market_2027)

# Market vs burn
cat_comp_2027['market_minus_burn'] = (
    cat_comp_2027['market_eur_2027'] - cat_comp_2027['mean_burn']
)
cat_comp_2027['market_over_burn'] = (
    cat_comp_2027['market_eur_2027'] / cat_comp_2027['mean_burn']
)

print("CAT 2027 vs burn (€/contract):")
print(cat_comp_2027[
    ['month_name', 'n_years', 'mean_burn', 'market_eur_2027',
     'market_minus_burn', 'market_over_burn', 'p10', 'p90', 'max']
])


CAT 2027 vs burn (€/contract):
      month_name  n_years    mean_burn  market_eur_2027  market_minus_burn  \
month                                                                        
4          April       30    20.166667           7040.0        7019.833333   
5            May       30   216.766667           9360.0        9143.233333   
6            Jun       30   959.700000          11520.0       10560.300000   
7            Jul       30  1737.600000          13060.0       11322.400000   
8            Aug       30  1635.333333          12840.0       11204.666667   
9            Sep       30   473.766667          10440.0        9966.233333   
10       October       30    49.900000           8360.0        8310.100000   

       market_over_burn    p10     p90     max  
month                                           
4            349.090909    0.0    37.9   256.0  
5             43.180071   28.4   488.9   721.0  
6             12.003751  486.9  1749.6  2217.0  
7              7.5161